# Stage 1: Data Lake Foundation
## ISM 6562 — Final Project: Sportlytics Athletics (Project 07)

**Team:** BigQuest Query  
**Stage:** 1 of 4 — Data Lake Foundation  
**Concepts:** HDFS architecture, zone design, file formats, replication strategy

---

### Business Context

Sportlytics Athletics is a sports analytics company that needs a scalable big data infrastructure to support:
- **Player performance tracking** across games and training sessions
- **Injury prediction and prevention** using historical injury reports
- **Game outcome analysis** using detailed game statistics
- **Schedule optimization** based on team performance trends

This notebook documents the design and setup of the HDFS data lake that forms the foundation of our big data pipeline.

---
## 1. Data Lake Architecture

We designed a **three-zone data lake** on HDFS. The zone names were adapted to better reflect the data lifecycle for a sports analytics context:

| Zone | HDFS Path | Purpose | Equivalent Term |
|------|-----------|---------|----------------|
| **Raw** | `/sportlytics/raw/` | Stores original ingested files, unchanged | Landing Zone |
| **Processed** | `/sportlytics/processed/` | Cleaned, transformed, joined datasets | Curated Zone |
| **Analytics** | `/sportlytics/analytics/` | Aggregated, query-ready outputs | Analytics Zone |

> **Note on naming:** We chose `raw`, `processed`, and `analytics` over the generic `landing`, `curated`, `analytics` naming because it more clearly communicates data state to the analytics team. The zones serve the same purpose and follow the same architectural pattern.

### Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                        HDFS DATA LAKE                               │
│                                                                     │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────────────┐  │
│  │  RAW ZONE    │───▶│ PROCESSED    │───▶│   ANALYTICS ZONE     │  │
│  │              │    │    ZONE      │    │                      │  │
│  │ /raw/        │    │ /processed/  │    │ /analytics/          │  │
│  │              │    │              │    │                      │  │
│  │ • CSV files  │    │ • Parquet    │    │ • Aggregated Parquet │  │
│  │ • JSON files │    │ • Cleaned    │    │ • Partitioned        │  │
│  │ • Original   │    │ • Joined     │    │ • BI-ready           │  │
│  │   format     │    │ • Validated  │    │                      │  │
│  │              │    │              │    │                      │  │
│  │ Replication=2│    │ Replication=2│    │ Replication=2        │  │
│  └──────────────┘    └──────────────┘    └──────────────────────┘  │
│                                                                     │
│  Infrastructure: 1 NameNode + 3 DataNodes (Hadoop 3.2.1)           │
└─────────────────────────────────────────────────────────────────────┘
```

---
## 2. Infrastructure Setup

The HDFS cluster is containerized using Docker Compose and consists of:

| Component | Container | Role |
|-----------|-----------|------|
| NameNode | `sportlytics-namenode` | Manages metadata, namespace, and block mapping |
| DataNode 1 | `sportlytics-datanode1` | Stores actual data blocks |
| DataNode 2 | `sportlytics-datanode2` | Stores actual data blocks |
| DataNode 3 | `sportlytics-datanode3` | Stores actual data blocks |
| ResourceManager | `sportlytics-resourcemanager` | YARN job scheduling |
| NodeManagers (x3) | `sportlytics-nodemanager1/2/3` | YARN task execution |

All nodes use the `tcsmith314/hadoop-*-python311` images (Hadoop 3.2.1, Python 3.11) as required by the course.

---
## 3. Creating the HDFS Zone Structure

The following commands were executed inside the NameNode container via the shell script `scripts/01_load_hdfs.sh` to create the three zones:

In [ ]:
import subprocess

# Display the zone creation commands
zone_commands = [
    "hdfs dfs -mkdir -p /sportlytics/raw",
    "hdfs dfs -mkdir -p /sportlytics/processed",
    "hdfs dfs -mkdir -p /sportlytics/analytics",
]

print("=== HDFS Zone Creation Commands ===")
for cmd in zone_commands:
    print(f"  $ {cmd}")
print("\nZones created successfully.")

### Verification — HDFS Zone Structure

After running the script, the zone listing confirmed all three directories were created:

```
$ docker exec -it sportlytics-namenode hdfs dfs -ls /sportlytics

Found 3 items
drwxr-xr-x   - root supergroup   0  2026-04-26  /sportlytics/analytics
drwxr-xr-x   - root supergroup   0  2026-04-26  /sportlytics/processed
drwxr-xr-x   - root supergroup   0  2026-04-26  /sportlytics/raw
```

---
## 4. Datasets

Five datasets were provided by the course data repository for the Sportlytics Athletics scenario. All files were loaded into the **Raw Zone** (`/sportlytics/raw/`).

| # | File | Format | Size | Description |
|---|------|--------|------|-------------|
| 1 | `player-tracking.csv` | CSV | ~55.8 MB | GPS/sensor tracking data per player per play |
| 2 | `game-stats.json` | JSON | ~72.0 MB | Detailed per-game statistics (nested structure) |
| 3 | `injury-reports.csv` | CSV | ~0.9 MB | Historical injury records per player |
| 4 | `training-sessions.json` | JSON | ~28.5 MB | Training load and performance metrics |
| 5 | `team-schedules.csv` | CSV | ~0.7 MB | Game schedules and opponent data |

---
## 5. File Format Justification

### Raw Zone — Preserve Original Formats

In the Raw Zone, we deliberately **kept the original file formats** (CSV and JSON). This is a best practice in data lake design because:
- The raw zone serves as an immutable source of truth
- If processing errors occur downstream, we can always reprocess from the original
- No transformation risk is introduced at ingestion time

| File | Format Choice | Justification |
|------|--------------|---------------|
| `player-tracking.csv` | CSV | Tabular, flat schema — CSV is ideal; easy to inspect and widely compatible |
| `game-stats.json` | JSON | Nested/hierarchical game data with variable fields — JSON preserves the structure natively |
| `injury-reports.csv` | CSV | Simple tabular records — CSV is lightweight and efficient for small files |
| `training-sessions.json` | JSON | Semi-structured training metrics with nested session details — JSON is the right fit |
| `team-schedules.csv` | CSV | Simple flat schedule table — CSV is appropriate for small, structured data |

### Processed Zone — Convert to Parquet

In Stage 2, all data will be converted to **Parquet** format because:
- Columnar storage allows Spark to skip columns not needed in a query
- Built-in compression (Snappy) significantly reduces storage size
- Schema enforcement catches data quality issues early
- Efficient for analytical aggregations (sum, avg, group by)

### Analytics Zone — Partitioned Parquet

Analytics outputs in Stage 2 will use **partitioned Parquet**, partitioned by relevant dimensions (e.g., by `season` or `team_id`) to reduce query scan size for downstream analysis.

---
## 6. Loading Data into the Raw Zone

All five files were loaded into `/sportlytics/raw/` using HDFS `put` commands inside the shell script:

In [ ]:
# Display the data loading commands used in the shell script
load_commands = [
    ("player-tracking.csv",    "hdfs dfs -put /data/player-tracking.csv    /sportlytics/raw/"),
    ("game-stats.json",        "hdfs dfs -put /data/game-stats.json        /sportlytics/raw/"),
    ("injury-reports.csv",     "hdfs dfs -put /data/injury-reports.csv     /sportlytics/raw/"),
    ("training-sessions.json", "hdfs dfs -put /data/training-sessions.json /sportlytics/raw/"),
    ("team-schedules.csv",     "hdfs dfs -put /data/team-schedules.csv     /sportlytics/raw/"),
]

print("=== Data Loading Commands ===")
for filename, cmd in load_commands:
    print(f"  [{filename}]")
    print(f"    $ {cmd}")
print("\nAll 5 files loaded successfully into /sportlytics/raw/")

### Verification — Files in Raw Zone

```
$ docker exec -it sportlytics-namenode hdfs dfs -ls /sportlytics/raw/

Found 5 items
-rw-r--r--   2 root supergroup   75510032  2026-04-26 15:52  /sportlytics/raw/game-stats.json
-rw-r--r--   2 root supergroup     940717  2026-04-26 15:52  /sportlytics/raw/injury-reports.csv
-rw-r--r--   2 root supergroup   58581296  2026-04-26 15:52  /sportlytics/raw/player-tracking.csv
-rw-r--r--   2 root supergroup     758354  2026-04-26 15:52  /sportlytics/raw/team-schedules.csv
-rw-r--r--   2 root supergroup   29900563  2026-04-26 15:52  /sportlytics/raw/training-sessions.json
```

All 5 files are present with the correct sizes. The `2` in the third column confirms the **replication factor of 2** is applied to every file.

---
## 7. Replication Strategy

All files in the raw zone are stored with a **replication factor of 2**, meaning each data block has 2 copies distributed across the 3 DataNodes.

### Why Replication Factor = 2?

| Factor | Tradeoff |
|--------|----------|
| Factor = 1 | No redundancy — single node failure causes data loss |
| **Factor = 2**  | One copy can fail; still accessible. Good balance for a course environment |
| Factor = 3 | Maximum fault tolerance (Hadoop default) — but uses 3x storage |

We chose **replication = 2** because:
- We have exactly 3 DataNodes — factor 2 distributes copies across at least 2 of them
- It provides fault tolerance if one DataNode goes down without tripling storage usage
- For a course environment, factor 3 would be excessive given our storage constraints

In a production system with higher data criticality (e.g., injury records used for legal purposes), we would increase replication to 3 for those specific files.

### Verification — Replication Factor Check

```
$ docker exec -it sportlytics-namenode hdfs dfs -stat "%r %n" /sportlytics/raw/*

2 game-stats.json
2 injury-reports.csv
2 player-tracking.csv
2 team-schedules.csv
2 training-sessions.json
```

All files confirmed at replication factor = 2.

### Verification — FSCK Health Check (player-tracking.csv)

```
$ docker exec -it sportlytics-namenode hdfs fsck /sportlytics/raw/player-tracking.csv -files -blocks

/sportlytics/raw/player-tracking.csv 58581296 bytes, replicated: replication=2, 1 block(s): OK

Status: HEALTHY
  Minimally replicated blocks:  1 (100.0%)
  Under-replicated blocks:      0 (0.0%)
  Missing blocks:               0
  Corrupt blocks:               0
  Average block replication:    2.0

The filesystem under path '/sportlytics/raw/player-tracking.csv' is HEALTHY
```

---
## 8. Cluster Health Report

```
$ docker exec -it sportlytics-namenode hdfs dfsadmin -report

Configured Capacity:  3,243,303,530,496 bytes (2.95 TB)
DFS Used:               334,135,296 bytes (318.66 MB)
DFS Remaining:        3,015,149,608,960 bytes (2.74 TB)

Live datanodes (3):
  datanode1 — 102.09 MB used, 3 blocks, HEALTHY
  datanode2 —  86.73 MB used, 4 blocks, HEALTHY
  datanode3 — 129.84 MB used, 3 blocks, HEALTHY

Under replicated blocks: 0
Missing blocks:          0
Corrupt blocks:          0
```

All 3 DataNodes are live and healthy. Blocks are distributed across all nodes with no replication or corruption issues.

---
## 9. Stage 1 Summary

| Requirement | Status | Details |
|-------------|--------|---------|
| HDFS zone structure created | **Done** | `raw`, `processed`, `analytics` zones under `/sportlytics/` |
| Raw datasets loaded | **Done** | All 5 files loaded into `/sportlytics/raw/` |
| File formats justified | **Done** | CSV and JSON preserved in raw zone; Parquet planned for processed/analytics |
| Replication factor set | **Done** | Factor = 2 on all files; justified for 3-DataNode cluster |
| HDFS health verified | **Done** | FSCK reports HEALTHY; 0 missing/corrupt blocks |
| Architecture documented | **Done** | Zone diagram and infrastructure table in Section 1–2 |

### Next Steps (Stage 2)
- Read raw files from `/sportlytics/raw/` using PySpark
- Clean, validate, and join datasets
- Write curated outputs to `/sportlytics/processed/` as Parquet
- Write aggregated analytics outputs to `/sportlytics/analytics/` with partitioning